In [1]:
import random
import pandas as pd

def generate_labeled_aml_samples_final(n_samples=1000, seed=42):
    random.seed(seed)

    # 1. Tính số lượng mẫu theo tỷ lệ và đảm bảo tổng = n_samples
    raw_ratios = {
        "Rất thấp": 0.30,
        "Thấp": 0.30,
        "Trung bình": 0.20,
        "Cao": 0.10,
        "Rất cao": 0.10,
    }

    def calculate_label_targets(n):
        label_counts = {k: int(n * v) for k, v in raw_ratios.items()}
        total_assigned = sum(label_counts.values())
        remaining = n - total_assigned
        sorted_labels = sorted(raw_ratios.items(), key=lambda x: -x[1])
        for i in range(remaining):
            label_counts[sorted_labels[i % len(sorted_labels)][0]] += 1
        return label_counts

    labels_target = calculate_label_targets(n_samples)

    # 2. Dữ liệu giả lập cơ bản
    residence_area = {
        'high': ["TP Hồ Chí Minh", "Hà Nội", "Hải Phòng", "Quảng Ninh", "Đà Nẵng",
                 "Lạng Sơn", "Lào Cai", "Tây Ninh", "Kiên Giang", "Bình Dương",
                 "Đồng Nai", "Cần Thơ", "An Giang", "Bà Rịa - Vũng Tàu"],
        'medium': ["Bình Thuận", "Bình Phước", "Bình Định", "Khánh Hòa", "Nghệ An",
                   "Thanh Hóa", "Thừa Thiên Huế", "Đắk Lắk", "Đắk Nông", "Gia Lai",
                   "Kon Tum", "Quảng Nam", "Quảng Ngãi", "Phú Yên", "Hà Tĩnh",
                   "Hải Dương", "Nam Định", "Ninh Bình", "Thái Nguyên", "Vĩnh Phúc",
                   "Long An", "Hậu Giang", "Tiền Giang", "Trà Vinh", "Vĩnh Long",
                   "Sóc Trăng", "Cà Mau", "Bạc Liêu", "Bến Tre", "Phú Thọ",
                   "Hưng Yên", "Thái Bình", "Ninh Thuận", "Hòa Bình", "Yên Bái",
                   "Tuyên Quang", "Hà Nam", "Lâm Đồng"],
        'low': ["Bắc Giang", "Bắc Kạn", "Bắc Ninh", "Cao Bằng", "Điện Biên",
                "Hà Giang", "Sơn La", "Lai Châu", "Quảng Trị", "Đồng Tháp"]
    }

    occupation = {
        'high': ["Cầm đồ", "Kinh doanh nhà hàng karaoke", "Chủ quán bar", "Làm từ thiện",
                 "Tiếp viên quán", "Vũ công tự do", "Streamer", "Youtuber", "Tiktoker",
                 "Kinh doanh vàng bạc", "Chơi chứng khoán", "Doanh nhân", "Tự doanh",
                 "Môi giới bất động sản", "Kinh doanh đa cấp", "Không rõ"],
        'medium': ["Luật sư", "Nhân viên ngân hàng", "Kế toán", "Kiểm toán viên",
                   "Chuyên viên tài chính", "Tư vấn bảo hiểm", "Môi giới chứng khoán",
                   "Tài xế giao dịch tiền", "Nhà báo", "Nghệ sĩ tự do", "Ca sĩ", "Diễn viên",
                   "MC", "Freelancer", "Nhà văn", "Nhiếp ảnh gia", "Thiết kế đồ họa",
                   "Chuyên gia SEO", "Quản trị fanpage", "Lái xe công nghệ", "Bán hàng online",
                   "Chủ cửa hàng", "Lái taxi", "Chủ quán ăn", "Nhân viên marketing",
                   "Cửa hàng trưởng", "Quản lý khách sạn", "Làm thuê thời vụ", "Trình dược viên",
                   "Sinh viên", "Thất nghiệp", "Nội trợ"],
        'low': ["Công an", "Bộ đội", "Thẩm phán", "Kiểm sát viên", "Giảng viên đại học",
                "Giáo viên phổ thông", "Gia sư", "Nhà khoa học", "Kỹ sư phần mềm", "Lập trình viên",
                "Kỹ sư xây dựng", "Kỹ sư điện", "Kỹ thuật viên phòng lab", "Chuyên viên CNTT",
                "Phân tích dữ liệu", "Bác sĩ", "Y tá", "Dược sĩ", "Bác sĩ thú y", 
                "Nhân viên chăm sóc sắc đẹp", "Chuyên viên spa", "Chăm sóc người già", "Công nhân",
                "Thợ xây", "Thợ điện", "Thợ nước", "Thợ mộc", "Thợ hàn", "Lái xe tải", "Bốc vác",
                "Bảo vệ", "Nhân viên bảo trì", "Nhân viên bán hàng", "Nhân viên phục vụ", "Lễ tân",
                "Nhân viên thu ngân", "Tư vấn tuyển sinh", "Học sinh"]
    }

    per_violation_score = {
        "Rửa tiền": 6, "Lừa đảo": 4, "Vi phạm dân sự": 2,
        "Không có hành vi phạm được đề cập": 0
    }

    org_violation_score = {
        "Đưa vào danh sách trừng phạt": 5,
        "Trừng phạt ngành lĩnh vực": 4,
        "Trừng phạt thứ cấp": 3,
        "Không có hành vi phạm được đề cập": 0
    }

    per_role_score = {
        "Chủ mưu": 6, "Tham gia": 4, "Bên liên quan bị động": 2,
        "Không có liên quan rõ ràng trong bài báo": 0
    }

    legal_status_score = {
        "Đã kết án": 3, "Đang điều tra": 2,
        "Tin chưa rõ ràng": 1, "Được minh oan": -999
    }

    def weighted_choice(dic):
        return random.choice(dic['high'] + dic['medium'] + dic['low'])

    def generate_sample():
        res_area = weighted_choice(residence_area)
        job = weighted_choice(occupation)
        alias_count = random.randint(0, 5)
        age = random.randint(18, 80)
        per_violation = random.choice(list(per_violation_score.keys()))
        per_role = random.choice(list(per_role_score.keys()))
        per_legal = random.choice(list(legal_status_score.keys()))
        org_violation = random.choice(list(org_violation_score.keys()))

        score = 0
        score += {"high": 2, "medium": 1, "low": 0}[[k for k, v in residence_area.items() if res_area in v][0]]
        score += {"high": 2, "medium": 0, "low": 0}[[k for k, v in occupation.items() if job in v][0]]
        score += 2 if alias_count > 1 else 0
        score += 1 if age < 30 or age > 65 else 0

        per_score = per_violation_score[per_violation]
        role_score = per_role_score[per_role]
        legal_score = legal_status_score[per_legal]
        org_score = org_violation_score[org_violation]

        per_cleared = legal_score == -999
        org_cleared = org_violation == "Không có hành vi phạm được đề cập"

        if per_cleared and org_cleared:
            per_score = role_score = legal_score = org_score = 0
        elif per_cleared:
            per_score = role_score = legal_score = 0
        elif org_cleared:
            org_score = 0
            legal_score = max(0, legal_score)
        else:
            legal_score = max(0, legal_score)

        score += per_score + role_score + legal_score + org_score

        if score >= 17:
            label = "Rất cao"
        elif score >= 13:
            label = "Cao"
        elif score >= 9:
            label = "Trung bình"
        elif score >= 4:
            label = "Thấp"
        else:
            label = "Rất thấp"

        return {
            "residence_area": res_area,
            "occupation": job,
            "alias_count": alias_count,
            "age": age,
            "per_violation_type": per_violation,
            "per_role": per_role,
            "per_legal_status": per_legal,
            "org_violation_type": org_violation,
            "total_score": score,
            "label": label
        }

    data = []
    counts = {k: 0 for k in labels_target}
    while sum(counts.values()) < n_samples:
        sample = generate_sample()
        label = sample["label"]
        if counts[label] < labels_target[label]:
            data.append(sample)
            counts[label] += 1

    return pd.DataFrame(data)

# ✅ Gọi hàm với số lượng bạn muốn
df = generate_labeled_aml_samples_final(n_samples=100)

# ✅ Kiểm tra phân phối
print(df["label"].value_counts(normalize=True).sort_index())


label
Cao           0.1
Rất cao       0.1
Rất thấp      0.3
Thấp          0.3
Trung bình    0.2
Name: proportion, dtype: float64


In [2]:
file = generate_labeled_aml_samples_final(n_samples=100)

In [3]:
file.head(50)

,residence_area,occupation,alias_count,age,per_violation_type,per_role,per_legal_status,org_violation_type,total_score,label
0,Cà Mau,Kinh doanh đa cấp,0,65,Vi phạm dân sự,Tham gia,Đang điều tra,Trừng phạt ngành lĩnh vực,15,Cao
1,Hòa Bình,Môi giới bất động sản,5,65,Rửa tiền,Không có liên quan rõ ràng trong bài báo,Đã kết án,Đưa vào danh sách trừng phạt,19,Rất cao
2,Lạng Sơn,Diễn viên,1,50,Rửa tiền,Tham gia,Được minh oan,Trừng phạt ngành lĩnh vực,6,Thấp
3,Hà Tĩnh,Thợ hàn,2,69,Rửa tiền,Tham gia,Được minh oan,Trừng phạt thứ cấp,7,Thấp
4,Khánh Hòa,Kiểm toán viên,1,79,Vi phạm dân sự,Chủ mưu,Đã kết án,Không có hành vi phạm được đề cập,13,Cao
5,Lào Cai,Sinh viên,2,56,Vi phạm dân sự,Chủ mưu,Được minh oan,Đưa vào danh sách trừng phạt,9,Trung bình
6,Lai Châu,Công an,0,53,Vi phạm dân sự,Bên liên quan bị động,Đang điều tra,Đưa vào danh sách trừng phạt,11,Trung bình
7,Hải Phòng,Tư vấn tuyển sinh,1,67,Vi phạm dân sự,Chủ mưu,Đang điều tra,Đưa vào danh sách trừng phạt,18,Rất cao
8,Kon Tum,Lái xe công nghệ,3,58,Vi phạm dân sự,Tham gia,Tin chưa rõ ràng,Trừng phạt thứ cấp,13,Cao
9,Bà Rịa - Vũng Tàu,Học sinh,2,62,Rửa tiền,Tham gia,Đang điều tra,Trừng phạt ngành lĩnh vực,20,Rất cao
